This notebook uploads data from a .csv file into the database.

1. **Imports** relevant libraries and sets up helpers.
2. **Event data** - change as needed.
3. **Prints** details of all items for confirmation before uploading.
4. **Resets event** to a blank event with no items.
5. **Uploads** all item data to the database.

In [ ]:
import os
from dotenv import load_dotenv
from supabase import create_client, Client
import pandas as pd
from datetime import date

load_dotenv()

supabase: Client = create_client(os.getenv("SUPABASE_URL"), os.getenv("SUPABASE_KEY"))

ref = {
    "vox": "vocals",
    "gt": "guitar",
    "keys": "keyboard",
    "bass": "bass guitar",
    "drums": "drum set",
    "vln": "violin",
    "vla": "viola",
    "vlc": "cello",
    "picc": "piccolo",
    "fl": "flute",
    "ob": "oboe",
    "cl": "clarinet",
    "bsn": "bassoon",
    "sax": "saxophone",
    "tpt": "trumpet",
    "tbn": "trombone",
    "glock": "glockenspiel",
    "s": "vocals",
    "a": "vocals",
    "t": "vocals",
    "b": "vocals"
}

def remove_numbers(s):
    res = ""
    for c in s:
        if not 48 <= ord(c) <= 57:
            res += c
    return res

In [ ]:
# Change this cell as needed

event_name = "Event name"
event_date = date(year, month, day)
internal = True / False
items = pd.read_csv("event_name.csv")

In [ ]:
first = True
broken = False
for _, row in items.iterrows():
    for col, val in row.items():
        if col == "song":
            if first:
                first = False
                print(f"song: {val}")
            else:
                print(f"\nsong: {val}")
        elif pd.notna(val):
            name = supabase.table("members").select("name").eq("telegram_handle", val).execute().data
            try:
                name = name[0]["name"]
                print(f"{name}: {ref[remove_numbers(col)]}")
            except IndexError:
                print(f"{val}: telehandle not found")
                broken = True
                break
    if broken:
        break

In [ ]:
supabase.table("events").delete().eq("name", event_name).execute()
supabase.table("events").insert({
    "name": event_name,
    "date": event_date.isoformat(),
    "internal": internal
}).execute()

In [ ]:
for _, row in items.iterrows():
    broken = False
    song = row["song"]
    try:
        result = supabase.table("items").upsert({
            "song": song,
            "event": event_name
        }, on_conflict="song,event").execute()
    except Exception as e:
        print(song, e, "\n")
    item_id = result.data[0]["id"]
    for col, val in row.items():
        if remove_numbers(col) in ref and pd.notna(val):
            try:
                supabase.table("players").upsert({
                "item_id": item_id,
                "telegram_handle": val,
                "role": ref[remove_numbers(col)]
            }, on_conflict="item_id,telegram_handle,role").execute()
            except Exception as e:
                print(val, e, "\n")
print("Done")